In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import lightgbm as lgb
import catboost as cb
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ADVANCED PRICE PREDICTION MODEL")
print("="*80)

# ============================================================================
# 1. LOAD DATA
# ============================================================================
print("\n[1/7] Loading data...")
train_df = pd.read_csv('student_resource/dataset/train.csv')
test_df = pd.read_csv('student_resource/dataset/test.csv')

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# ============================================================================
# 2. ADVANCED FEATURE ENGINEERING
# ============================================================================
print("\n[2/7] Engineering features...")

def extract_features(df):
    """Extract comprehensive features from catalog content"""
    
    # Basic text stats
    df['text_length'] = df['catalog_content'].str.len()
    df['word_count'] = df['catalog_content'].str.split().str.len()
    df['avg_word_length'] = df['text_length'] / (df['word_count'] + 1)
    df['uppercase_ratio'] = df['catalog_content'].str.count(r'[A-Z]') / (df['text_length'] + 1)
    df['digit_ratio'] = df['catalog_content'].str.count(r'\d') / (df['text_length'] + 1)
    df['punctuation_ratio'] = df['catalog_content'].str.count(r'[.,!?;:]') / (df['text_length'] + 1)
    
    # Price signals (HIGH IMPACT based on EDA)
    df['has_warranty'] = df['catalog_content'].str.contains(r'warranty|guarantee', case=False, na=False).astype(int)
    df['has_dimensions'] = df['catalog_content'].str.contains(r'\d+\s*x\s*\d+|\d+"\s*x\s*\d+"', case=False, na=False).astype(int)
    df['has_weight'] = df['catalog_content'].str.contains(r'\d+\s*(lb|lbs|kg|ounce|oz|pound)', case=False, na=False).astype(int)
    df['has_material'] = df['catalog_content'].str.contains(r'(steel|plastic|wood|metal|aluminum|leather|fabric|cotton)', case=False, na=False).astype(int)
    df['has_brand'] = df['catalog_content'].str.contains(r'brand|branded', case=False, na=False).astype(int)
    df['has_color'] = df['catalog_content'].str.contains(r'(black|white|red|blue|green|yellow|silver|gold|gray|brown)', case=False, na=False).astype(int)
    
    # Premium indicators
    df['has_premium_words'] = df['catalog_content'].str.contains(
        r'premium|professional|deluxe|quality|best|top|high-end|luxury', 
        case=False, na=False
    ).astype(int)
    
    # Discount/budget indicators
    df['has_budget_words'] = df['catalog_content'].str.contains(
        r'budget|cheap|affordable|value|economy|basic', 
        case=False, na=False
    ).astype(int)
    
    # Product category hints
    df['is_electronic'] = df['catalog_content'].str.contains(
        r'electronic|digital|battery|power|usb|wireless|bluetooth', 
        case=False, na=False
    ).astype(int)
    
    df['is_clothing'] = df['catalog_content'].str.contains(
        r'shirt|pants|dress|jacket|shoes|clothing|apparel|wear', 
        case=False, na=False
    ).astype(int)
    
    df['is_home_goods'] = df['catalog_content'].str.contains(
        r'kitchen|home|furniture|decor|bedding|towel|curtain', 
        case=False, na=False
    ).astype(int)
    
    # Numbers in text (often correlate with specifications)
    df['number_count'] = df['catalog_content'].str.findall(r'\d+').str.len()
    df['has_large_number'] = df['catalog_content'].str.contains(r'\d{3,}', na=False).astype(int)
    
    # Extract numeric values (sizes, weights, etc.)
    df['max_number'] = df['catalog_content'].str.findall(r'\d+').apply(
        lambda x: max([int(i) for i in x]) if x and len(x) > 0 else 0
    )
    
    # Sentence structure
    df['sentence_count'] = df['catalog_content'].str.count(r'[.!?]') + 1
    df['avg_sentence_length'] = df['word_count'] / df['sentence_count']
    
    # Special characters
    df['has_trademark'] = df['catalog_content'].str.contains(r'™|®|©', na=False).astype(int)
    df['has_measurements'] = df['catalog_content'].str.contains(r'\d+\s*(inch|cm|mm|ft|yard|meter)', case=False, na=False).astype(int)
    
    return df

train_df = extract_features(train_df)
test_df = extract_features(test_df)

print(f"Created {len([c for c in train_df.columns if c not in ['sample_id', 'catalog_content', 'image_link', 'price']])} features")

# ============================================================================
# 3. TF-IDF TEXT FEATURES
# ============================================================================
print("\n[3/7] Extracting TF-IDF features...")

# Use bigrams and trigrams for better context
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
    strip_accents='unicode',
    lowercase=True,
    analyzer='word',
    token_pattern=r'\w{2,}',
    stop_words='english'
)

# Fit on all data
all_text = pd.concat([
    train_df['catalog_content'], 
    test_df['catalog_content']
])

tfidf_matrix_train = tfidf.fit_transform(train_df['catalog_content'])
tfidf_matrix_test = tfidf.transform(test_df['catalog_content'])

# Reduce dimensionality with SVD
svd = TruncatedSVD(n_components=100, random_state=42)
tfidf_train_svd = svd.fit_transform(tfidf_matrix_train)
tfidf_test_svd = svd.transform(tfidf_matrix_test)

# Add to dataframe
for i in range(tfidf_train_svd.shape[1]):
    train_df[f'tfidf_svd_{i}'] = tfidf_train_svd[:, i]
    test_df[f'tfidf_svd_{i}'] = tfidf_test_svd[:, i]

print(f"Added {tfidf_train_svd.shape[1]} TF-IDF SVD components")

# ============================================================================
# 4. TARGET TRANSFORMATION (CRITICAL for skewed data)
# ============================================================================
print("\n[4/7] Applying log transformation to target...")

# Log transform (based on EDA: skewness=13.6)
train_df['log_price'] = np.log1p(train_df['price'])

print(f"Original price - Mean: ${train_df['price'].mean():.2f}, Std: ${train_df['price'].std():.2f}")
print(f"Log price - Mean: {train_df['log_price'].mean():.4f}, Std: {train_df['log_price'].std():.4f}")

# ============================================================================
# 5. PREPARE FEATURES
# ============================================================================
print("\n[5/7] Preparing feature matrices...")

# Select feature columns
feature_cols = [col for col in train_df.columns if col not in [
    'sample_id', 'catalog_content', 'image_link', 'price', 'log_price'
]]

X_train = train_df[feature_cols].copy()
y_train = train_df['log_price'].values
X_test = test_df[feature_cols].copy()

print(f"Feature matrix shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")

# ============================================================================
# 6. TRAIN MODELS WITH K-FOLD CV
# ============================================================================
print("\n[6/7] Training models with 5-fold CV...")

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store predictions
oof_lgb = np.zeros(len(train_df))
oof_cat = np.zeros(len(train_df))
predictions_lgb = np.zeros(len(test_df))
predictions_cat = np.zeros(len(test_df))

# SMAPE metric
def smape(y_true, y_pred):
    y_true = np.expm1(y_true)  # Convert back from log
    y_pred = np.expm1(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0.0
    return 100 * np.mean(diff)

fold_scores_lgb = []
fold_scores_cat = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f"\n--- Fold {fold}/{n_folds} ---")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # ========== LightGBM ==========
    print("Training LightGBM...")
    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.03,
        'num_leaves': 63,
        'max_depth': 8,
        'min_child_samples': 20,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    
    model_lgb = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )
    
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    predictions_lgb += model_lgb.predict(X_test) / n_folds
    
    smape_lgb = smape(y_val, oof_lgb[val_idx])
    fold_scores_lgb.append(smape_lgb)
    print(f"LightGBM SMAPE: {smape_lgb:.4f}")
    
    # ========== CatBoost ==========
    print("Training CatBoost...")
    cat_params = {
        'loss_function': 'RMSE',
        'learning_rate': 0.03,
        'depth': 8,
        'l2_leaf_reg': 3,
        'random_seed': 42,
        'verbose': False,
        'iterations': 2000,
        'early_stopping_rounds': 100,
        'task_type': 'GPU',  # Use your GPU!
        'devices': '0'
    }
    
    model_cat = cb.CatBoostRegressor(**cat_params)
    model_cat.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        verbose=False
    )
    
    oof_cat[val_idx] = model_cat.predict(X_val)
    predictions_cat += model_cat.predict(X_test) / n_folds
    
    smape_cat = smape(y_val, oof_cat[val_idx])
    fold_scores_cat.append(smape_cat)
    print(f"CatBoost SMAPE: {smape_cat:.4f}")

print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS")
print("="*80)
print(f"LightGBM  - Mean SMAPE: {np.mean(fold_scores_lgb):.4f} (+/- {np.std(fold_scores_lgb):.4f})")
print(f"CatBoost  - Mean SMAPE: {np.mean(fold_scores_cat):.4f} (+/- {np.std(fold_scores_cat):.4f})")

# ============================================================================
# 7. ENSEMBLE & GENERATE PREDICTIONS
# ============================================================================
print("\n[7/7] Creating ensemble and generating predictions...")

# Weighted ensemble (adjust weights based on CV performance)
lgb_weight = 1.0 / np.mean(fold_scores_lgb)
cat_weight = 1.0 / np.mean(fold_scores_cat)
total_weight = lgb_weight + cat_weight

predictions_ensemble = (
    (predictions_lgb * lgb_weight + predictions_cat * cat_weight) / total_weight
)

# OOF ensemble for final CV score
oof_ensemble = (
    (oof_lgb * lgb_weight + oof_cat * cat_weight) / total_weight
)

final_smape = smape(y_train, oof_ensemble)
print(f"\nFinal Ensemble SMAPE: {final_smape:.4f}")

# Convert back from log space
final_predictions = np.expm1(predictions_ensemble)

# Ensure non-negative predictions
final_predictions = np.maximum(final_predictions, 0.01)

# ============================================================================
# 8. SAVE PREDICTIONS
# ============================================================================
print("\nSaving predictions...")

submission = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_predictions
})

submission.to_csv('submission.csv', index=False)
print(f"Saved to submission.csv")

print("\n" + "="*80)
print("PREDICTION STATISTICS")
print("="*80)
print(f"Mean predicted price: ${submission['price'].mean():.2f}")
print(f"Median predicted price: ${submission['price'].median():.2f}")
print(f"Min predicted price: ${submission['price'].min():.2f}")
print(f"Max predicted price: ${submission['price'].max():.2f}")

print("\n" + "="*80)
print("✓ MODEL TRAINING COMPLETE!")
print("="*80)
print(f"Expected SMAPE: ~{final_smape:.2f}")
print("Submit 'submission.csv' to the portal!")